# Session 2: Spatial autocorrelation, diffusion, and a capstone

Same graph as Session 1. Nothing here rebuilds it from scratch.

In [ ]:
# Colab only. Locally this is a no-op and prints nothing.
# Identical to the first cell of Session 1: on Colab it installs the spatial stack and
# downloads the two data files, on your own machine DATA already points at them.
import pathlib
import subprocess
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/farrencc/dist_net_finder/main/graph-workshop"
DATA = pathlib.Path("../data")

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "geopandas==1.1.4", "libpysal==4.14.1", "esda==2.9.0", "splot==1.1.7",
         "osmnx==2.1.1", "mapclassify==2.10.0"],
        check=True,
    )
    DATA = pathlib.Path("data")
    DATA.mkdir(exist_ok=True)
    for fname in ("dublin_eds.gpkg", "streets.graphml"):
        urllib.request.urlretrieve(f"{REPO_RAW}/data/{fname}", DATA / fname)
    print("Colab setup done:", sorted(p.name for p in DATA.iterdir()))

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from libpysal.weights import Queen
from scipy.linalg import eigh

# Same seeds and the same attribute as Session 1, so the numbers line up.
SEED = 7
RNG = np.random.default_rng(20240402)
ATTR = "pct_third_level"

plt.rcParams.update({
    "figure.figsize": (9.0, 5.0),
    "figure.dpi": 110,
    "font.size": 13,
    "axes.titlesize": 15,
    "axes.labelsize": 13,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "lines.linewidth": 2.2,
    "lines.markersize": 7,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "figure.constrained_layout.use": True,
})

## 0. Reload

`session1_state.npz` comes from Session 1, section 7. If it is missing, the two label
vectors are recomputed here instead and everything else is identical either way. The
cell prints which of the two it used.

In [ ]:
# Rebuild the Session 1 objects: the EDs, the Queen contiguity graph, the standardised
# attribute, and the full eigendecomposition of the unweighted Laplacian.
gdf = gpd.read_file(DATA / "dublin_eds.gpkg", layer="eds").to_crs(2157)
w = Queen.from_dataframe(gdf, use_index=False)
G = w.to_networkx()

n = len(gdf)
pts = gdf.geometry.representative_point()
POS = {i: (pts.iloc[i].x, pts.iloc[i].y) for i in range(n)}
nx.set_node_attributes(G, dict(zip(range(n), gdf["ED_ENGLISH"])), "name")

y = gdf[ATTR].to_numpy(float)
z = (y - y.mean()) / y.std(ddof=0)

# The full spectrum, not just lambda_2: section 3 needs every mode to run the diffusion.
L = nx.laplacian_matrix(G, nodelist=range(n)).toarray().astype(float)
evals, evecs = eigh(L)

cache = DATA / "session1_state.npz"
if cache.exists():
    state = np.load(cache, allow_pickle=True)
    SIGMA = float(state["sigma"])
    labels_unweighted = state["labels_unweighted"]
    labels_weighted = state["labels_weighted"]
    source = "session1_state.npz"
else:
    diffs = np.array([abs(z[i] - z[j]) for i, j in G.edges()])
    SIGMA = float(np.median(diffs))
    Wm = np.zeros((n, n))
    for i, j in G.edges():
        Wm[i, j] = Wm[j, i] = np.exp(-((z[i] - z[j]) ** 2) / (2 * SIGMA ** 2))
    Dm = np.diag(Wm.sum(axis=1))
    labels_unweighted = (evecs[:, 1] > 0).astype(int)
    labels_weighted = (eigh(Dm - Wm, Dm)[1][:, 1] > 0).astype(int)
    source = "recomputed (no cache found)"

print(f"{n} EDs, {G.number_of_edges()} contiguity edges, connected={nx.is_connected(G)}")
print(f"lambda_2 = {evals[1]:.5f}")
print(f"Session 1 labels from: {source}   sigma = {SIGMA:.3f}")
print("spot check:", [(i, G.nodes[i]['name'], gdf['ED_ENGLISH'].iloc[i]) for i in (0, 80, n - 1)])

## 1. Recap (0:00)

Slides.

## 2. Moran's I (0:10)

$I = \dfrac{n}{\sum_{ij} w_{ij}} \cdot \dfrac{\sum_{ij} w_{ij} z_i z_j}{\sum_i z_i^2}$,
with $z$ the centred attribute.

In [ ]:
# A spatial weights matrix and an adjacency matrix are the same object under two names.
# 'b' is binary (the raw adjacency), 'r' is row-standardised so each row sums to 1,
# which is what Moran's I expects.
A = nx.adjacency_matrix(G, nodelist=range(n)).toarray()

w.transform = "b"
print("unstandardised W is the Session 1 adjacency matrix:", np.array_equal(w.full()[0].astype(int), A))

w.transform = "r"
print("row sums after 'r':", np.unique(np.round(w.full()[0].sum(axis=1), 12)))

In [ ]:
from esda import Moran

PERMUTATIONS = 999
ALPHA = 0.05

# Moran's I: the correlation between each ED's value and the average of its neighbours.
# Significance comes from permutations, not from a normal table, because the values are
# not independent draws and no closed-form null applies.
mi = Moran(y, w, permutations=PERMUTATIONS)
print(f"I      = {mi.I:+.4f}")
print(f"E[I]   = {mi.EI:+.4f}   (= -1/(n-1) = {-1 / (n - 1):+.4f}, not zero)")
print(f"z-score= {mi.z_sim:+.2f}")
print(f"p_sim  = {mi.p_sim:.4f}   ({PERMUTATIONS} permutations, alpha = {ALPHA})")

In [ ]:
# The null distribution, built by shuffling the same values over the same map 999 times.
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.hist(mi.sim, bins=40, color="0.6", edgecolor="white")
ax.axvline(mi.EI, color="0.2", ls="--", lw=2, label=rf"$E[I] = {mi.EI:+.4f}$")
ax.axvline(mi.I, color="crimson", lw=3, label=rf"observed $I = {mi.I:+.4f}$")
ax.set_xlabel("Moran's I under random reassignment")
ax.set_ylabel("permutations")
ax.set_title(f"{PERMUTATIONS} permutations of the same values over the same map")
ax.legend()
plt.show()

In [ ]:
# Each ED plotted against the average of its neighbours. The slope of the fit IS I.
from splot.esda import moran_scatterplot

fig, ax = plt.subplots(figsize=(7.5, 7))
moran_scatterplot(mi, ax=ax, aspect_equal=False)
ax.set_xlabel("z")
ax.set_ylabel("spatial lag of z")
ax.annotate(rf"slope = $I$ = {mi.I:.3f}", xy=(0.05, 0.92), xycoords="axes fraction",
            fontsize=15, color="crimson")
plt.show()

In [ ]:
from esda import Moran_Local

# Moran's I was one test on the whole map. LISA is one test per ED, so 162 tests,
# and about 8 of them will look significant at alpha = 0.05 through luck alone.
lisa = Moran_Local(y, w, permutations=PERMUTATIONS, seed=SEED)


# Benjamini-Hochberg controls the false discovery rate: sort the p-values and keep
# everything up to the largest one still under alpha * rank / m.
def benjamini_hochberg(p, alpha):
    order = np.argsort(p)
    m = p.size
    passed = p[order] <= alpha * np.arange(1, m + 1) / m
    keep = np.zeros(m, dtype=bool)
    if passed.any():
        keep[order[: np.max(np.nonzero(passed)) + 1]] = True
    return keep


raw_sig = lisa.p_sim < ALPHA
bh_sig = benjamini_hochberg(lisa.p_sim, ALPHA)

print(f"significant at alpha = {ALPHA}          : {raw_sig.sum():3d} / {n}")
print(f"after Benjamini-Hochberg (FDR = {ALPHA}) : {bh_sig.sum():3d} / {n}")
print(f"after Bonferroni (alpha/n = {ALPHA / n:.2e}) : {(lisa.p_sim < ALPHA / n).sum():3d} / {n}")
# A permutation p-value has a hard floor at 1/(permutations + 1). Bonferroni at this n
# is asking for a number that 999 permutations simply cannot produce, so the zero it
# reports is a property of the test setup and not a statement about Dublin.
print(f"smallest p_sim {PERMUTATIONS} permutations can produce : {1 / (PERMUTATIONS + 1):.2e}")

In [ ]:
# HH and LL are clusters, HL and LH are spatial outliers: one ED unlike its neighbours.
from splot.esda import lisa_cluster

fig, ax = plt.subplots(figsize=(8, 9))
lisa_cluster(lisa, gdf, p=ALPHA, ax=ax, legend_kwds={"loc": "lower right"})
ax.set_axis_off()
ax.set_title(f"LISA clusters, p < {ALPHA}")
plt.show()

In [ ]:
# The corrected LISA map beside the Session 1 weighted clusters. Two different methods,
# same graph, same attribute: worth seeing where they agree and where they do not.
QUAD = {0: "not sig", 1: "HH", 2: "LH", 3: "LL", 4: "HL"}
quad = np.where(bh_sig, lisa.q, 0)

fig, axes = plt.subplots(1, 2, figsize=(15, 8))
gdf.assign(_q=[QUAD[v] for v in quad]).plot(column="_q", categorical=True, cmap="Set1",
                                            legend=True, edgecolor="white", linewidth=0.3,
                                            ax=axes[0], legend_kwds={"loc": "lower right"})
axes[0].set_title("LISA, BH-corrected")
gdf.assign(_l=labels_weighted).plot(column="_l", categorical=True, cmap="Set2",
                                    edgecolor="white", linewidth=0.3, ax=axes[1])
axes[1].set_title(rf"Session 1 weighted clusters, $\sigma$ = {SIGMA:.2f}")
for ax in axes:
    ax.set_axis_off()
plt.show()

In [ ]:
# The same agreement as a table rather than as two maps.
print(pd.crosstab(pd.Series(labels_weighted, name="cluster"),
                  pd.Series([QUAD[v] for v in quad], name="LISA")))

## 3. Diffusion (0:45)

$\dfrac{du_i}{dt} = \sum_{j \sim i} (u_j - u_i)$, which is $\dot{u} = -Lu$.

In [ ]:
# Sanity check before trusting anything below: the plain-English rule "each ED moves
# towards the average of its neighbours", written as a loop, is exactly -L u.
u_test = RNG.normal(size=n)
by_hand = np.array([sum(u_test[j] - u_test[i] for j in G[i]) for i in range(n)])

print("max |loop - (-L u)| =", np.abs(by_hand - (-L @ u_test)).max())

In [ ]:
# Release the mass at the best-connected ED, so the test is as favourable as possible
# to the mass getting out. Its Fiedler entry tells us which side of the line it starts on.
betweenness = nx.betweenness_centrality(G)
SHOCK = max(betweenness, key=betweenness.get)

print(f"shock at node {SHOCK}: {G.nodes[SHOCK]['name']}")
print(f"  betweenness {betweenness[SHOCK]:.4f} (rank 1 of {n}), degree {G.degree(SHOCK)}")
print(f"  Fiedler entry {evecs[SHOCK, 1]:+.4f}")

In [ ]:
# TAU = 1/lambda_2 is the natural clock of the graph: the decay time of the slowest
# non-constant mode. Times are quoted as multiples of it so they transfer to other graphs.
TAU = 1.0 / evals[1]
TIMES = [m * TAU for m in (0.02, 0.05, 0.12, 0.30, 1.00)]

u0 = np.zeros(n)
u0[SHOCK] = 1.0
coeffs = evecs.T @ u0


# Solve u' = -L u exactly rather than stepping it: expand the initial state in the
# eigenvectors, multiply each coefficient by exp(-lambda_k t), and transform back.
# No timestep, no stability condition, valid at any t.
def diffuse(source_node, t):
    """State of the graph heat equation at time t, unit mass released at source_node."""
    c = evecs.T[:, source_node]
    return evecs @ (np.exp(-evals * t) * c)


print(f"tau = 1/lambda_2 = {TAU:.2f}")
for t in TIMES:
    u = diffuse(SHOCK, t)
    print(f"  t = {t:8.3f}   sum u = {u.sum():.12f}   max u = {u.max():.4f}")

In [ ]:
# Each panel is scaled to its own maximum, so these show where the front has reached
# rather than how much is left. The totals are all exactly 1, as printed above.
# The cyan outline is the source ED.
fig, axes = plt.subplots(1, len(TIMES), figsize=(4 * len(TIMES), 5.5))
for ax, t in zip(axes, TIMES):
    gdf.assign(_u=diffuse(SHOCK, t)).plot(column="_u", cmap="inferno", ax=ax,
                                          edgecolor="white", linewidth=0.15)
    gdf.iloc[[SHOCK]].plot(ax=ax, facecolor="none", edgecolor="cyan", linewidth=2.5)
    ax.set_axis_off()
    ax.set_title(rf"$t$ = {t / TAU:.2f}$\tau$")
plt.show()

In [ ]:
# The same solution taken apart mode by mode: each one is a coefficient times
# exp(-lambda_k t), so large eigenvalues die first and lambda_1 = 0 never dies at all.
t_grid = np.geomspace(0.01 * TAU, 8 * TAU, 200)
amps = np.abs(coeffs[None, :] * np.exp(-evals[None, :] * t_grid[:, None]))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(t_grid, amps[:, 2:], color="0.75", lw=1.0)
ax.plot(t_grid, amps[:, 1], color="crimson", lw=3, label=rf"$\lambda_2$ = {evals[1]:.4f}")
ax.plot(t_grid, amps[:, 0], color="k", lw=3, ls="--", label=r"$\lambda_1 = 0$: the conserved mean")
ax.axvline(TAU, color="0.3", ls=":", lw=2)
ax.text(TAU * 1.1, 2e-6, r"$t = 1/\lambda_2$", fontsize=13)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(1e-6, 1.2)
ax.set_xlabel("t")
ax.set_ylabel(r"$|c_k e^{-\lambda_k t}|$")
ax.set_title(r"every mode decays except $\lambda_1$, and $\lambda_2$ decays slowest")
ax.legend(loc="lower left")
plt.show()

In [ ]:
# Three routes to the same state: the matrix exponential, our eigendecomposition,
# and explicit Euler with a safe timestep. They should agree to near machine precision.
from scipy.linalg import expm

t_check = 0.5 * TAU
u_expm = expm(-L * t_check) @ u0
u_eig = diffuse(SHOCK, t_check)

dt = 0.5 * 2.0 / evals[-1]
u_euler = u0.copy()
for _ in range(int(round(t_check / dt))):
    u_euler = u_euler - dt * (L @ u_euler)

print(f"expm vs eigendecomposition : {np.abs(u_expm - u_eig).max():.3e}")
print(f"expm vs Euler (dt = {dt:.4f}) : {np.abs(u_expm - u_euler).max():.3e}")

In [ ]:
# Explicit Euler on u' = -L u is stable only for dt < 2/lambda_max. Since
# lambda_max <= 2 * max degree, the graph itself dictates the largest usable timestep.
dt_max = 2.0 / evals[-1]
print(f"lambda_max      = {evals[-1]:.3f}")
print(f"2 * max degree  = {2 * max(d for _, d in G.degree())}   (the bound lambda_max cannot exceed)")
print(f"stable dt       < {dt_max:.4f}")

fig, ax = plt.subplots(figsize=(9, 5.5))
for factor, style in [(0.9, "-"), (1.01, "--")]:
    step = factor * dt_max
    u = u0.copy()
    trace = []
    for _ in range(400):
        u = u - step * (L @ u)
        trace.append(np.abs(u).max())
    ax.semilogy(trace, style, label=rf"$\Delta t$ = {factor:.2f} $\times$ 2/$\lambda_{{max}}$")
ax.set_xlabel("Euler step")
ax.set_ylabel(r"$\max_i |u_i|$")
ax.set_title("explicit Euler, either side of the stability threshold")
ax.legend()
plt.show()

In [ ]:
# The late-time snapshot with the Session 1 Fiedler boundary drawn over it.
# This is the capstone question posed as a picture: has the mass crossed the cyan line?
fig, ax = plt.subplots(figsize=(8, 9))
gdf.assign(_u=diffuse(SHOCK, 1.0 * TAU)).plot(column="_u", cmap="inferno", ax=ax,
                                              edgecolor="white", linewidth=0.15)
gdf[evecs[:, 1] > 0].dissolve().boundary.plot(ax=ax, color="cyan", linewidth=2.5)
gdf.iloc[[SHOCK]].plot(ax=ax, facecolor="none", edgecolor="white", linewidth=2.5)
ax.set_axis_off()
ax.set_title(r"$t = 1/\lambda_2$, with the Fiedler boundary drawn on top")
plt.show()

## 4. Capstone (1:35)

### The question

In Session 1 you drew a line across Dublin using nothing but the second eigenvector of a
weighted Laplacian. It is a boundary in the sense that an algorithm chose to cut there.
Whether it is a boundary in any physical sense is a different question, and it is the one
you are answering now: **if something spreads across the contiguity graph, does that line
slow it down?**

### What you already have in memory

`diffuse(source_node, t)` releases one unit of mass at a single ED and returns where that
mass has reached at time `t`, using the eigendecomposition from section 3. Total mass is
conserved, so whatever time you ask for, the 162 numbers add to 1.

`labels_unweighted` and `labels_weighted` are the two Session 1 partitions of the same 162
EDs, each an array of 0s and 1s. The first used geometry only, the second used geometry
and the census attribute together.

`SHOCK` is the highest-betweenness ED, the natural place to release the mass.

`TAU`, `evals`, `evecs`, `G`, `gdf` and `n` are all still live from section 3.

### What you are being asked to build

Four steps, in the code cell below.

**1. Pick a time.** Leakage measured at t = 0 is zero and leakage measured at very large t
is whatever the equilibrium happens to be, so the answer depends entirely on when you
look. Choose one time and be ready to say why that one.

**2. Measure the real boundary.** Run the diffusion from `SHOCK` to your chosen time, then
use `leakage()` to get the fraction of mass that has ended up on the far side of the
Session 1 partition. That single number is your observed statistic. Do it for
`labels_weighted`, and for `labels_unweighted` too if you have time.

**3. Build a null.** One number on its own says nothing: 0.32 is neither large nor small
until there is something to compare it with. Generate many alternative boundaries that
match the real one in every respect you do not care about (the same graph, the same
source, the same number of EDs on the far side) and differ in the one respect you do
(they are not the spectral cut). Measure leakage across each of them in exactly the same
way. `random_contiguous_partition()` is there to draw them.

**4. Compare and report.** Where does the observed number sit inside the null
distribution? Count how many null draws leak less than the real boundary does. That rank
gives a p-value of `(rank + 1) / (draws + 1)`. Then fill in the conclusion cell below.

### What a good answer looks like

Not a small p-value. A good answer is one where you can state what your null holds fixed
and what it varies, and where the summary you quote matches the shape of the distribution
it came from. If the null comes out skewed, a z-score will flatter you and a percentile
will not.

Work on your own. The hints below are in order, so open them one at a time. The two
functions in the next cell are the whole toolkit.

In [ ]:
def leakage(u, labels, side):
    """Mass sitting on the far side of the partition, given the source's side.

    `side` is the label of the side the source started on, so pass labels[SHOCK].
    Total mass is 1, so the answer is directly the fraction that crossed.
    """
    return float(u[labels != side].sum())


def random_contiguous_partition(G, size, rng=RNG, max_tries=500):
    """A connected set of exactly `size` nodes, grown from a random seed.

    Picks a start node at random, then repeatedly absorbs a random node from
    the current frontier. Retries from a new seed if the frontier ever empties
    before the region is full.
    """
    nodes = list(G)
    if not 0 < size <= len(nodes):
        raise ValueError(f"size must be in 1..{len(nodes)}, got {size}")
    for _ in range(max_tries):
        region = {nodes[rng.integers(len(nodes))]}
        frontier = set(G[next(iter(region))])
        while len(region) < size and frontier:
            pick = sorted(frontier)[rng.integers(len(frontier))]
            region.add(pick)
            frontier.discard(pick)
            frontier |= set(G[pick]) - region
        if len(region) == size:
            return np.array(sorted(region))
    raise RuntimeError(f"could not grow a connected region of {size} in {max_tries} tries")


# Already in scope, nothing to rebuild: diffuse(source_node, t), G, gdf, evals, evecs,
# TAU, labels_unweighted, labels_weighted, SHOCK, n

# Everything below this line is a self-test of the two functions, so you can see what
# they return before you build anything on them.

_probe = [random_contiguous_partition(G, 40) for _ in range(200)]
print("200 draws of size 40:")
print("  all the right size :", all(len(r) == 40 for r in _probe))
print("  all connected      :", all(nx.is_connected(G.subgraph(r)) for r in _probe))
print("  distinct regions   :", len({tuple(r) for r in _probe}), "of 200")
print("  EDs never included :", n - len(set(np.concatenate(_probe))))
print("  cluster sizes     :", np.bincount(labels_weighted))

_demo = np.zeros(n, dtype=int)
_demo[random_contiguous_partition(G, 40)] = 1
_u = diffuse(SHOCK, 0.1 * TAU)
print(f"leakage() over one such region, either way round, at t = 0.1 tau:"
      f" {leakage(_u, _demo, 0):.4f} + {leakage(_u, _demo, 1):.4f} = {_u.sum():.4f}")

<details><summary><b>Hint 1</b></summary>

Total `u` is conserved and starts at 1, so "how much crossed" is a sum over the far side.

</details>

<details><summary><b>Hint 2</b></summary>

Why $t = 1/\lambda_2$ and not any other time? Look again at the modal amplitude
plot in section 3 and ask what is still alive at that moment.

</details>

<details><summary><b>Hint 3</b></summary>

For a null: keep the cluster sizes, scatter the labels randomly across the map.

</details>

<details><summary><b>Hint 4</b></summary>

That null is weak. Scattered labels form no boundary at all, so almost anything beats
it. Grow a region out from a random seed to the same size instead, and you are comparing
against a real but arbitrary division of the city.

</details>

In [ ]:
# ===========================================================================
# YOUR WORK
#
# Step 1. Pick a time and say why.
#   t = ...

# Step 2. Measure the real partition.
#   Run diffuse() from SHOCK to your time, then leakage() against labels_weighted.
#   Remember leakage() needs the source's own side, i.e. labels[SHOCK].

# Step 3. Build a null.
#   Draw many regions of the same size as the far side, measure each one the same way,
#   and collect the results in an array.

# Step 4. Compare.
#   rank = number of null draws that leak LESS than the real boundary
#   p    = (rank + 1) / (draws + 1)
#   Then plot the null distribution with the observed value marked on it.
# ===========================================================================

### Conclusion

**What I measured:**

**What I compared it against:**

**What I found:**

**What I would check next:**

### Angle B: does any of this survive losing the geography?

Same values, same map, shuffled between EDs.

In [ ]:
# Keep the 162 values and the 162 polygons, and break the pairing between them.
# Anything that survives this was never about geography in the first place.
shuffled = RNG.permutation(y)
zs = (shuffled - shuffled.mean()) / shuffled.std(ddof=0)


def weighted_split(values, sigma):
    Wm = np.zeros((n, n))
    for i, j in G.edges():
        Wm[i, j] = Wm[j, i] = np.exp(-((values[i] - values[j]) ** 2) / (2 * sigma ** 2))
    Dm = np.diag(Wm.sum(axis=1))
    return (eigh(Dm - Wm, Dm)[1][:, 1] > 0).astype(int)


mi_shuffled = Moran(shuffled, w, permutations=PERMUTATIONS)
labels_shuffled = weighted_split(zs, SIGMA)

print(f"Moran's I   before {mi.I:+.4f} (p = {mi.p_sim:.3f})   after {mi_shuffled.I:+.4f} (p = {mi_shuffled.p_sim:.3f})")
print(f"cluster sizes before {np.bincount(labels_weighted)}   after {np.bincount(labels_shuffled)}")

fig, axes = plt.subplots(2, 2, figsize=(13, 14))
for ax, col, cmap, title in [
    (axes[0, 0], y, "viridis", "attribute"),
    (axes[0, 1], shuffled, "viridis", "attribute, shuffled"),
]:
    gdf.assign(_v=col).plot(column="_v", cmap=cmap, ax=ax, edgecolor="white", linewidth=0.2)
    ax.set_title(title)
for ax, lab, title in [
    (axes[1, 0], labels_weighted, "weighted clusters"),
    (axes[1, 1], labels_shuffled, "weighted clusters, shuffled"),
]:
    gdf.assign(_l=lab).plot(column="_l", categorical=True, cmap="Set2", ax=ax,
                            edgecolor="white", linewidth=0.2)
    ax.set_title(title)
for ax in axes.ravel():
    ax.set_axis_off()
plt.show()

## 5. Wrap-up (2:15)

Slides.